In [0]:
# =============================================================================
# ICEBASE — PHASE 4 | NOTEBOOK 01
# Feature Engineering — Build the Fan Feature Table
# Idaho Mashers Hockey Analytics Platform
# =============================================================================
# STANDALONE NOTEBOOK — Run attached to a DEDICATED (single-user) cluster.
#
# ⚠️ CRITICAL COMPUTE REQUIREMENT:
#   ML workloads using Feature Engineering in Unity Catalog REQUIRE a cluster
#   with Access Mode = "Dedicated" (formerly "Single User").
#   Shared or No Isolation clusters will throw permission errors.
#   See the Phase 4 checklist Step A-2 for cluster creation instructions.
#
# WHAT THIS NOTEBOOK DOES:
#   Reads from icebase.gold.customer_360 and icebase.gold.retention_cohort,
#   computes a clean set of ML-ready RFM features, and writes them to a
#   Unity Catalog Feature Table at icebase.gold.fan_features.
#
#   This feature table is then used by BOTH ML models in Phase 4:
#     - Notebook 02: K-Means fan segmentation (unsupervised)
#     - Notebook 03: XGBoost churn prediction (supervised)
#
# WHY A FEATURE TABLE?
#   Centralizes feature computation in one place. Both models read from
#   the same features so there's no training/serving skew. If customer_360
#   is refreshed, re-running this notebook updates the feature table and
#   both models benefit automatically.
#
# TABLE WRITTEN:
#   icebase.gold.fan_features  (Unity Catalog Feature Table)
#
# FEATURES COMPUTED:
#   RFM core:     recency_days, frequency_games, monetary_total
#   Behavioral:   promo_sensitivity, advance_purchase_rate, avg_days_before_game
#   Loyalty:      tenure_days, is_season_holder, channels_used
#   Event:        jersey_night_attendee, hot_start_buyer, slump_buyer
#   Churn signal: churn_flag, days_since_last (from retention_cohort)
# =============================================================================

In [0]:
# COMMAND ----------
# ── CELL 1: Install and Import ─────────────────────────────────────────────
# Feature Engineering client requires databricks-feature-engineering package.
# Pre-installed on Databricks Runtime for ML — install manually if using standard DBR.
 
%pip install databricks-feature-engineering --quiet
dbutils.library.restartPython()

In [0]:
# COMMAND ----------
 
import mlflow
import pandas as pd
from pyspark.sql import functions as F
from databricks.feature_engineering import FeatureEngineeringClient
 
# Point MLflow at Unity Catalog registry
mlflow.set_registry_uri("databricks-uc")
 
CATALOG       = "icebase"
GOLD          = f"{CATALOG}.gold"
FEATURE_TABLE = f"{GOLD}.fan_features"
 
fe = FeatureEngineeringClient()
 
print("✅ Feature Engineering client initialized")
print(f"   Feature table target: {FEATURE_TABLE}")

In [0]:
# COMMAND ----------
# ── CELL 2: Read Gold Sources ──────────────────────────────────────────────
 
c360     = spark.table(f"{GOLD}.customer_360")
cohort   = spark.table(f"{GOLD}.retention_cohort")
 
print(f"customer_360 rows:     {c360.count():,}")
print(f"retention_cohort rows: {cohort.count():,}")

In [0]:
# COMMAND ----------
# ── CELL 3: Build Fan Feature DataFrame ───────────────────────────────────
#
# All features must be numeric or boolean for ML models.
# Categorical columns (acquisition_channel, initial_segment) are
# excluded from this feature table — the models use behavioral signals,
# not demographic labels. This prevents the models from learning segment
# labels that would create circular logic in the scoring output.
#
# customer_id is the PRIMARY KEY — required for Feature Engineering tables.
# It is the join key used at inference time.
 
features = (
    c360.select(
        # Primary key — required for Feature Engineering
        F.col("customer_id"),
 
        # ── RFM Features ────────────────────────────────────────────────
        # Recency: days since last purchase (lower = more engaged)
        F.coalesce(F.col("days_since_last"), F.lit(999)).alias("recency_days"),
 
        # Frequency: number of distinct games attended
        F.coalesce(F.col("games_attended"), F.lit(0)).alias("frequency_games"),
 
        # Monetary: total ticket spend (net of discounts)
        F.coalesce(F.col("revenue_net"), F.lit(0.0)).alias("monetary_net"),
 
        # Raw gross spend (before discounts — useful for promo ROI models)
        F.coalesce(F.col("total_spend"), F.lit(0.0)).alias("monetary_gross"),
 
        # ── Promo Behavior ──────────────────────────────────────────────
        # 0.0 = never uses promos, 1.0 = always uses deep discounts
        F.coalesce(F.col("promo_sensitivity"), F.lit(0.0)).alias("promo_sensitivity"),
 
        # Raw promo rate (fraction of tickets that were promo)
        F.coalesce(F.col("promo_rate"), F.lit(0.0)).alias("promo_rate"),
 
        # Total discount dollars received
        F.coalesce(F.col("total_discount_received"), F.lit(0.0)).alias("total_discount_received"),
 
        # ── Purchase Timing ─────────────────────────────────────────────
        # Fraction of purchases that were advance (>3 days before game)
        F.round(
            F.coalesce(F.col("advance_purchases"), F.lit(0)) /
            F.greatest(F.coalesce(F.col("total_tickets"), F.lit(1)), F.lit(1)),
            4
        ).alias("advance_purchase_rate"),
 
        # Average days before game purchased
        F.coalesce(F.col("avg_days_before_game"), F.lit(0.0)).alias("avg_days_before_game"),
 
        # ── Seat Preference ─────────────────────────────────────────────
        # Average seat tier rank (1=standing, 5=rinkside)
        F.coalesce(F.col("avg_seat_tier_rank"), F.lit(3.0)).alias("avg_seat_tier_rank"),
 
        # ── Loyalty Signals ─────────────────────────────────────────────
        F.coalesce(F.col("tenure_days"), F.lit(0)).alias("tenure_days"),
 
        # Season holder flag (1=yes, 0=no) — most loyal tier
        F.col("is_season_holder").cast("int").alias("is_season_holder"),
 
        # Number of different purchase channels used
        F.coalesce(F.col("channels_used"), F.lit(1)).alias("channels_used"),
 
        # ── Event Participation ──────────────────────────────────────────
        F.col("jersey_night_attendee").cast("int").alias("jersey_night_attendee"),
        F.col("hot_start_buyer").cast("int").alias("hot_start_buyer"),
        F.col("slump_buyer").cast("int").alias("slump_buyer"),
        F.col("late_push_buyer").cast("int").alias("late_push_buyer"),
        F.col("was_lapsed_reactivation").cast("int").alias("was_lapsed_reactivation"),
    )
    # Join churn signal from retention_cohort
    .join(
        cohort.select(
            "customer_id",
            F.col("churn_flag").cast("int").alias("churn_flag"),
            F.coalesce(F.col("days_since_last"), F.lit(999)).alias("cohort_days_since_last"),
            F.col("returned_30d").cast("int").alias("returned_30d"),
            F.col("returned_60d").cast("int").alias("returned_60d"),
            F.col("is_one_and_done").cast("int").alias("is_one_and_done"),
            F.col("is_jersey_night_cohort").cast("int").alias("is_jersey_night_cohort"),
        ),
        "customer_id",
        "left"
    )
    # Fill nulls from cohort join for customers with no cohort data
    .fillna({
        "churn_flag": 1,           # No cohort data = churned
        "cohort_days_since_last": 999,
        "returned_30d": 0,
        "returned_60d": 0,
        "is_one_and_done": 1,
        "is_jersey_night_cohort": 0,
    })
)
 
print(f"Feature DataFrame shape: {features.count():,} rows × {len(features.columns)} columns")
print(f"Columns: {features.columns}")

In [0]:
# COMMAND ----------
# ── CELL 4: Write to Feature Table ────────────────────────────────────────
#
# FeatureEngineeringClient.create_table() creates a managed Unity Catalog
# Delta table with primary key constraints tracked by the Feature Store.
# If the table already exists, write_table() with mode="overwrite" updates it.
 
try:
    # Try to create the table (first run)
    fe.create_table(
        name        = FEATURE_TABLE,
        primary_keys= ["customer_id"],
        df          = features,
        description = (
            "Idaho Mashers fan feature table — RFM + behavioral features "
            "for segmentation and churn prediction models. "
            "Refreshed from icebase.gold.customer_360 and retention_cohort."
        ),
    )
    print(f"✅ Feature table CREATED: {FEATURE_TABLE}")
 
except Exception as e:
    if "already exists" in str(e).lower():
        # Table exists — overwrite with latest features
        fe.write_table(
            name = FEATURE_TABLE,
            df   = features,
            mode = "overwrite",
        )
        print(f"✅ Feature table UPDATED: {FEATURE_TABLE}")
    else:
        raise e
 
row_count = spark.table(FEATURE_TABLE).count()
print(f"   Rows in feature table: {row_count:,}")

In [0]:
# COMMAND ----------
# ── CELL 5: Validate Feature Table ────────────────────────────────────────
 
print("── Feature summary statistics ──")
display(
    spark.table(FEATURE_TABLE)
    .select(
        "recency_days", "frequency_games", "monetary_net",
        "promo_sensitivity", "advance_purchase_rate",
        "tenure_days", "churn_flag"
    )
    .summary("count", "mean", "min", "25%", "75%", "max")
)
 
# Check for nulls — ML models cannot handle nulls
null_counts = {
    col: spark.table(FEATURE_TABLE)
              .filter(F.col(col).isNull())
              .count()
    for col in ["recency_days", "frequency_games", "monetary_net",
                "promo_sensitivity", "churn_flag"]
}
print(f"\n── Null check (all should be 0) ──")
for col, cnt in null_counts.items():
    status = "✅" if cnt == 0 else "❌"
    print(f"  {status} {col}: {cnt} nulls")
 
print(f"\n── Churn rate in feature table ──")
churn_rate = spark.table(FEATURE_TABLE).agg(
    F.round(F.avg("churn_flag") * 100, 1).alias("churn_rate_pct")
).collect()[0].churn_rate_pct
print(f"  Overall churn rate: {churn_rate}%")
print(f"  (Expected: 40–65% given mix of lapsed/casual segments in seed data)")